# INF2006 G12 — Task 3 step 0: volatility labels
Creates the target for `INF2006_G12_task3_volatility.ipynb` and writes it to `s3a://<bucket>/task3/volatility_v1/{training,validation,testing,metadata}`.

**Label:** `label_volatility_5d = 1` when the sample standard deviation of a stock's daily close-to-close returns over the **next five trading sessions** is above the **75th percentile of the training split** (threshold computed on 2013–2015 only). A row is kept only if all five future returns exist **and** the fifth future session falls inside the same split year, so no label crosses a split boundary. Future values are used only for the label, never as features.

**Run:** PySpark kernel, Kernel → Restart & Run All, before the volatility notebook. Expected: 349,590 / 122,801 / 123,476 rows; threshold ≈ 0.015929.

In [1]:
BUCKET = "inf2006-g12-ml"          # change to your bucket
BASE = "s3a://" + BUCKET
OUTPUT = BASE + "/task3/volatility_v1"
from pyspark.sql import functions as F
from pyspark.sql.window import Window

train = spark.read.parquet(BASE + "/task2/model_ready_v1/training")
val = spark.read.parquet(BASE + "/task2/model_ready_v1/validation")
test = spark.read.parquet(BASE + "/task2/model_ready_v1/testing")
prices = spark.read.parquet(BASE + "/task2/cleaned/prices")
counts = dict(training=train.count(), validation=val.count(), testing=test.count(), prices=prices.count())
print(counts)
assert counts == dict(training=349590, validation=122801, testing=126001, prices=605887), "Task 2 inputs changed"


Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,application_1795071391258_0001,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

{'training': 349590, 'validation': 122801, 'testing': 126001, 'prices': 605887}

In [2]:
# Future five-session volatility per ticker/date (label only, never a feature).
p = prices.select("ticker", F.to_date("trade_date").alias("trade_date"), F.col("close_price").cast("double").alias("close_price"))
assert p.groupBy("ticker", "trade_date").count().filter("count > 1").limit(1).count() == 0, "Duplicate ticker-date"
w = Window.partitionBy("ticker").orderBy("trade_date")
p = p.withColumn("prev_close", F.lag("close_price").over(w)) \
     .withColumn("ret", F.when(F.col("prev_close") > 0, F.col("close_price") / F.col("prev_close") - 1))
nxt = w.rowsBetween(1, 5)
future = (p.withColumn("future_volatility_5d", F.stddev_samp("ret").over(nxt))
           .withColumn("future_count", F.count("ret").over(nxt))
           .withColumn("future_end_date", F.lead("trade_date", 5).over(w))
           .filter((F.col("future_count") == 5) & F.col("future_volatility_5d").isNotNull())
           .select("ticker", "trade_date", "future_volatility_5d", "future_count", "future_end_date"))

def attach(df, last_day):
    out = df.withColumn("trade_date", F.to_date("trade_date")).join(future, ["ticker", "trade_date"], "inner")
    return out.filter(F.col("future_end_date") <= F.to_date(F.lit(last_day)))  # outcome must finish inside the split

train_v, val_v, test_v = attach(train, "2015-12-31"), attach(val, "2016-12-31"), attach(test, "2017-12-31")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
# Threshold from TRAINING only, then label all three splits with it.
threshold = float(train_v.approxQuantile("future_volatility_5d", [0.75], 0.001)[0])
print("High-volatility threshold (training 75th percentile):", threshold)
lab = lambda df: df.withColumn("label_volatility_5d", (F.col("future_volatility_5d") > threshold).cast("double"))
splits = dict(training=lab(train_v), validation=lab(val_v), testing=lab(test_v))
for name, df in splits.items():
    df.write.mode("overwrite").parquet(OUTPUT + "/" + name)
spark.createDataFrame([(threshold,)], ["high_volatility_threshold"]).write.mode("overwrite").json(OUTPUT + "/metadata")

expected = dict(training=349590, validation=122801, testing=123476)
for name in splits:
    saved = spark.read.parquet(OUTPUT + "/" + name)
    n, pos = saved.count(), saved.filter("label_volatility_5d = 1").count()
    print(name, "rows", n, "| class 1:", pos, "(" + str(round(pos / n, 4)) + ")")
    assert n == expected[name], "Unexpected row count for " + name
print("SAVED:", OUTPUT)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

High-volatility threshold (training 75th percentile): 0.01593094111871191
training rows 349590 | class 1: 87673 (0.2508)
validation rows 122801 | class 1: 39913 (0.325)
testing rows 123476 | class 1: 22376 (0.1812)
SAVED: s3a://inf2006-g12-ml/task3/volatility_v1